# Selected temporal model: train → resume → infer → evaluate

Select recurrent (ConvGRU), Mamba, native pair, or native pair with pretext losses.
Unlike the spatial-only SA fine-tuning notebook, this workflow dispatches to
`train_temporal_model` through the temporal CLI. It runs in one process on one
selected device; it does not force the spatial notebook's two-GPU FSDP setup.

Select one case/configuration below. Each action uses that configuration and its
selected checkpoint. Training and full-period inference run only when their
switches are enabled. Backend comparisons belong to the separate experiment CLI.

Use the **Prithvi** kernel. These are experimental workflows; passing an engineering
test does not establish scientific acceptance. NARR requires actual local data,
scalers and checkpoint paths. Existing Linux symlink stubs are preserved.


In [ ]:
from pathlib import Path
import json, subprocess, sys, yaml
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'granitewxc/temporal').is_dir())
CASE = 'SA'                    # 'SA' or 'NARR'
BACKEND = 'recurrent'          # 'recurrent', 'mamba', 'native_pair', 'native_pair_pretext'
CONFIGURATIONS = {'SA/recurrent': 'examples/CORDEX_ML/SA_downscaling_refinement_T2_ACCESS-CM2_static_temporal_recurrent.yaml', 'SA/mamba': 'examples/CORDEX_ML/SA_downscaling_refinement_T2_ACCESS-CM2_static_temporal_mamba.yaml', 'SA/native_pair': 'examples/CORDEX_ML/SA_downscaling_refinement_T2_ACCESS-CM2_static_temporal_prithvi_native_pair.yaml', 'SA/native_pair_pretext': 'examples/CORDEX_ML/SA_downscaling_refinement_T2_ACCESS-CM2_static_temporal_prithvi_native_pair_pretext.yaml', 'NARR/recurrent': 'examples/NARR_PRISM/NARR_PRISM_subdomain_temporal_recurrent.yaml', 'NARR/mamba': 'examples/NARR_PRISM/NARR_PRISM_subdomain_temporal_mamba.yaml', 'NARR/native_pair': 'examples/NARR_PRISM/NARR_PRISM_subdomain_temporal_prithvi_native_pair.yaml', 'NARR/native_pair_pretext': 'examples/NARR_PRISM/NARR_PRISM_subdomain_temporal_prithvi_native_pair_pretext.yaml'}
CONFIG = ROOT / CONFIGURATIONS[f'{CASE}/{BACKEND}']
# Native variants also offer sibling *_extended_v2.yaml training schedules.
RUN = ROOT / f'examples/{"CORDEX_ML" if CASE == "SA" else "NARR_PRISM"}/runs_temporal/selected_{CASE}_{BACKEND}_v2'
CHECKPOINT = RUN / 'checkpoints/last.ckpt'
CLI = ROOT / ('examples/CORDEX_ML/cordex_temporal_training.py' if CASE == 'SA' else 'examples/NARR_PRISM/narr_prism_temporal.py')
OVERRIDES = {}  # Existing dotted keys → values; see examples below.
# NARR examples (actual paths required):
# OVERRIDES = {'data.preprocessed_dir': '/data/preprocessed',
#              'temporal.init_from_spatial_checkpoint': '/data/checkpoints/last.ckpt',
#              'model.input_mu': '/data/scalars/inputs_mean.npy', ...}
DEVICE = None                 # None = YAML/auto; or 'cuda' / 'cpu'
UPDATES_PER_EPOCH = None       # Optional successful-update cap; None = full epoch
VALIDATION_WINDOWS = None      # None = full configured validation split
TOTAL_EPOCHS = None            # None = selected YAML's num_epochs
RESUME_TOTAL_EPOCHS = None     # Total desired epochs; None = selected YAML
RUN_TRAIN = True
RUN_RESUME = False
RUN_INFER = False
RUN_EVALUATE = False
assert CONFIG.is_file(), CONFIG
assert not (RUN_TRAIN and RUN_RESUME), 'Select train OR resume for this execution'
assert Path(sys.prefix).name.casefold() == 'prithvi', 'Select the Prithvi kernel before running this workflow'
print('Interpreter:', sys.executable)
print('Configuration:', CONFIG)
print('Checkpoint:', CHECKPOINT)


In [ ]:
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from granitewxc.temporal.progress import run_notebook_command
# Remember identical warnings across train/resume/infer and helper-cell reruns.
_TEMPORAL_SEEN_WARNINGS = globals().get('_TEMPORAL_SEEN_WARNINGS', set())

def invoke(action, *arguments):
    command = [sys.executable, '-u', str(CLI), action, '--config', str(CONFIG), '--notebook-output']
    if DEVICE is not None:
        command.extend(['--device', DEVICE])
    for key, value in OVERRIDES.items():
        command.extend(['--set', f'{key}={json.dumps(value)}'])
    command.extend(str(a) for a in arguments)
    print(subprocess.list2cmdline(command))
    run_notebook_command(command, cwd=ROOT, seen_warnings=_TEMPORAL_SEEN_WARNINGS)

def training_arguments(epochs):
    arguments = ['--num-workers', 0]
    for flag, value in (('--epochs', epochs), ('--max-steps', UPDATES_PER_EPOCH),
                        ('--max-val-steps', VALIDATION_WINDOWS)):
        if value is not None:
            arguments.extend([flag, value])
    return arguments


## Train the selected model
Defaults use the selected YAML epoch count, accumulation and complete data splits. Optional `--max-steps` caps successful optimizer updates per epoch. One tqdm bar per epoch updates after each training microbatch, showing loss and successful optimizer updates. The same bar shows validation status and its final loss. Identical warnings appear once per notebook session. Validation selects `best.ckpt`; `last.ckpt` preserves continuation state.

In [ ]:
if RUN_TRAIN:
    assert not CHECKPOINT.exists(), 'Choose a new RUN directory or use resume'
    invoke('train', '--output-dir', RUN / 'checkpoints', *training_arguments(TOTAL_EPOCHS),
           '--output', RUN / 'training_summary.json')


## Resume the selected checkpoint
Keep the data, loss, accumulation and step protocol identical for exact resume. `RESUME_TOTAL_EPOCHS` is the desired total; increase it explicitly to extend a completed run. The trainer rejects incompatible protocols. Legacy checkpoints lacking RNG and data order are explicitly reported as approximate restarts.

In [ ]:
if RUN_RESUME:
    assert CHECKPOINT.is_file(), CHECKPOINT
    invoke('train', '--resume', CHECKPOINT, '--output-dir', RUN / 'checkpoints',
           *training_arguments(RESUME_TOTAL_EPOCHS),
           '--output', RUN / 'resume_summary.json')


## Infer with the selected checkpoint
Defaults to the configured full test period. SA bounded comparison uses 1981-01-01 through 1983-12-31. Daily means remain daily means. Chunking supplies real history; unavailable history at a true run start follows the declared cold-start rule.

In [ ]:
if RUN_INFER:
    assert CHECKPOINT.is_file(), CHECKPOINT
    invoke('infer', '--checkpoint', CHECKPOINT, '--split', 'test', '--chunk-length', 30,
           '--output-dir', RUN / 'inference', '--output', RUN / 'inference_summary.json')


## Evaluate the emitted predictions
The summary identifies the exact prediction file. Metrics use matching timestamps, masks and physical units. The same test period must not become a repeated development target.

In [ ]:
if RUN_EVALUATE:
    summary = json.loads((RUN / 'inference_summary.json').read_text())
    invoke('evaluate', '--predictions', summary['npz'], '--event-aligned',
           '--output', RUN / 'evaluation.json')


## Separate backend comparison and refinement

The six-variant comparison is `cordex_temporal_experiment.py --steps 600
--val-steps 60 --epochs 1 --test-years 3 --out <new-directory>`; it is intentionally
not run by this notebook. Optional `*_extended_v2.yaml` schedules are separate
development experiments, not the pre-registered comparison.

Recurrent and Mamba entry points and all four stochastic-refinement interfaces
remain available. Interface compatibility does not show a refiner remains
calibrated after changing Phase-1 predictions. Refinement retraining is separate.
